# ELT Pipeline – Bronze Layer (Raw Ingestion)

**Author:** Jarosław Błaziński  
**Project:** E-Commerce Medallion Architecture (Databricks Community Edition)  
**Stack:** PySpark, Delta Tables, Databricks  

---

## Co robi ta warstwa?

Bronze = surowe dane bez żadnych zmian.  
Zasada: **load as-is** – zapisujemy wszystko co dostajemy ze źródła, łącznie z NULLami i błędami.  
Dzięki temu zawsze możemy wrócić do oryginału i sprawdzić co przyszło ze źródła.

```
CSV (source)
     ↓
[BRONZE]  ← jesteśmy tutaj
     ↓
[SILVER]
     ↓
[GOLD]
```

## 1. Konfiguracja

In [ ]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType, TimestampType
)

# Na Databricks SparkSession jest już dostępna jako 'spark'
# Poniższa linia działa też lokalnie
spark = SparkSession.builder.appName("ELT_Bronze").getOrCreate()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("BRONZE")

# ── Ścieżki ──────────────────────────────────────────────────
# Databricks Community Edition: wgraj plik przez Data > Add Data
# i podmień ścieżkę poniżej na: /FileStore/tables/input_data_v2.csv
INPUT_PATH  = "input_data_v2.csv"
BRONZE_PATH = "/tmp/medallion/bronze/transactions"

logger.info("Konfiguracja załadowana. Input: %s", INPUT_PATH)

## 2. Definicja schematu

Definiujemy schemat ręcznie zamiast używać `inferSchema=True`.  
Dlaczego? Bo `inferSchema` skanuje cały plik (wolno przy 500k wierszy)  
i może błędnie odgadnąć typy danych.

In [ ]:
SCHEMA = StructType([
    StructField("transaction_id",   IntegerType(),   nullable=False),
    StructField("customer_id",      IntegerType(),   nullable=True),
    StructField("amount",           DoubleType(),    nullable=True),   # NULL-e celowe
    StructField("category",         StringType(),    nullable=True),
    StructField("country",          StringType(),    nullable=True),
    StructField("status",           StringType(),    nullable=True),
    StructField("payment_method",   StringType(),    nullable=True),
    StructField("quantity",         IntegerType(),   nullable=True),
    StructField("transaction_date", TimestampType(), nullable=True),
])

## 3. Wczytanie danych z CSV

In [ ]:
df_raw = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(SCHEMA)
    .csv(INPUT_PATH)
)

row_count = df_raw.count()
logger.info("Wczytano %d wierszy z CSV", row_count)

print(f"Liczba wierszy : {row_count:,}")
print(f"Liczba kolumn  : {len(df_raw.columns)}")
df_raw.printSchema()

## 4. Dodanie metadanych ingestii

W Bronze dodajemy kolumny techniczne – kiedy dane zostały załadowane i skąd.  
To podstawa audit trail: zawsze wiesz kiedy i co trafiło do systemu.

In [ ]:
df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())   # kiedy załadowano
    .withColumn("_source_file", lit(INPUT_PATH))        # skąd pochodzi
)

df_bronze.show(5, truncate=False)

## 5. Zapis jako Delta Table

Delta format = Parquet + transaction log.  
Dzięki temu mamy: wersjonowanie danych, możliwość MERGE/UPDATE/DELETE,  
a później Time Travel (wrócimy do tego w Gold).

In [ ]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")          # przy kolejnym uruchomieniu nadpisuje
    .partitionBy("country")     # partycjonowanie przyspiesza filtry po country
    .save(BRONZE_PATH)
)

logger.info("Bronze Delta Table zapisana w: %s", BRONZE_PATH)
print(f"✅ Bronze zapisana: {BRONZE_PATH}")

## 6. Weryfikacja zapisu

Zawsze warto potwierdzić że Delta Table działa poprawnie po zapisie.

In [ ]:
df_verify = spark.read.format("delta").load(BRONZE_PATH)

print(f"Wierszy w Delta Table : {df_verify.count():,}")
print(f"Partycje (country)    : {df_verify.select('country').distinct().count()}")
df_verify.show(5, truncate=False)

## 7. Historia Delta Table (Time Travel)

Delta automatycznie zapisuje historię każdej operacji.  
Możemy zobaczyć wszystkie wersje tabeli i w razie potrzeby wrócić do poprzedniej.

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

## 8. Quality check – raport jakości surowych danych

Dokumentujemy co mamy NA WEJŚCIU – to ważne dla audytu i dla Silver layer,  
która będzie musiała te problemy naprawić.

In [ ]:
from pyspark.sql.functions import col, count, sum as spark_sum, round as spark_round

total = df_bronze.count()

quality_report = df_bronze.agg(
    count("*").alias("total_rows"),
    spark_sum(col("amount").isNull().cast("int")).alias("null_amounts"),
    spark_sum((col("amount") < 0).cast("int")).alias("negative_amounts"),
    spark_sum(col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
)

quality_report.show()

null_pct = df_bronze.filter(col("amount").isNull()).count() / total * 100
neg_pct  = df_bronze.filter(col("amount") < 0).count()     / total * 100

print(f"NULL amounts    : {null_pct:.1f}%")
print(f"Negative amounts: {neg_pct:.1f}%")
print()
print("Rozkład po krajach:")
df_bronze.groupBy("country").count().orderBy("country").show()
print("Rozkład po kategoriach:")
df_bronze.groupBy("category").count().orderBy("category").show()

## Podsumowanie Bronze Layer

| Krok | Status |
|---|---|
| Wczytanie CSV ze schematem | ✅ |
| Dodanie metadanych ingestii | ✅ |
| Zapis jako Delta Table (partitioned by country) | ✅ |
| Weryfikacja zapisu | ✅ |
| Raport jakości danych | ✅ |

**Następny krok:** `5_ELT_Silver.ipynb` – czyszczenie, transformacje, wzbogacanie danych.